#   Data Cleaning 



## Importing Libraries & Load Data

In [70]:
import pandas as pd

df = pd.read_csv("../Data/Nepal-roadaccidents.csv")
print(f"Loaded: {df.shape[0]} rows * {df.shape[1]} columns")
df.head(3)

Loaded: 550 rows * 22 columns


,Accident_ID,Year,Month,Province,District,City / Place,Latitude,Longitude,Road_Type,Vehicle_Type,...,Number_of_Vehicles_Involved,Injuries,Fatalities,Emergency_Response_Time_Minutes,Weather_Condition,SDG_Indicator,Victim_Gender,Victim_Age_Group,Road_User_Type,Total_People_Killed
0,ACC-00001,2022,12,Karnali,Surkhet,Birendranagar,28.616393,81.618594,Highway,Motorcycle,...,4,2,0,10,Clear,SDG 3.6,Male,25–44,Motorcyclist,0
1,ACC-00002,2025,7,Lumbini,Rupandehi,Gaidahawa,27.561930,83.398993,Urban Road,Motorcycle,...,2,5,3,26,Rainy,SDG 3.6,Female,45–64,Motorcyclist,3
2,ACC-00003,2023,10,Madhesh,Dhanusha,Dhanushadham,26.758865,85.925803,Urban Road,Motorcycle,...,4,0,0,58,Foggy,SDG 11.2,Female,25–44,Motorcyclist,0


## Understand Shape & Columns

In [71]:
print("Shape:", df.shape)
print("Column Names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2}. {col}")

Shape: (550, 22)
Column Names:
   1. Accident_ID
   2. Year
   3. Month
   4. Province
   5. District
   6. City / Place
   7. Latitude
   8. Longitude
   9. Road_Type
  10. Vehicle_Type
  11. Accident_Cause
  12. Time_of_Day
  13. Number_of_Vehicles_Involved
  14. Injuries
  15. Fatalities
  16. Emergency_Response_Time_Minutes
  17. Weather_Condition
  18. SDG_Indicator
  19. Victim_Gender
  20. Victim_Age_Group
  21. Road_User_Type
  22. Total_People_Killed


##  Check Data Types

In [72]:
df.dtypes

Accident_ID                            str
Year                                 int64
Month                                int64
Province                               str
District                               str
City / Place                           str
Latitude                           float64
Longitude                          float64
Road_Type                              str
Vehicle_Type                           str
Accident_Cause                         str
Time_of_Day                            str
Number_of_Vehicles_Involved          int64
Injuries                             int64
Fatalities                           int64
Emergency_Response_Time_Minutes      int64
Weather_Condition                      str
SDG_Indicator                          str
Victim_Gender                          str
Victim_Age_Group                       str
Road_User_Type                         str
Total_People_Killed                  int64
dtype: object

## Check for Missing Values

In [73]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing)
print(f"Total missing cells: {missing.sum()}")
print(f"Dataset completeness: {(1 - missing.sum()/(df.shape[0]*df.shape[1]))*100:.1f}%")

Missing values per column:
Accident_ID                        0
Year                               0
Month                              0
Province                           0
District                           0
City / Place                       0
Latitude                           0
Longitude                          0
Road_Type                          0
Vehicle_Type                       0
Accident_Cause                     0
Time_of_Day                        0
Number_of_Vehicles_Involved        0
Injuries                           0
Fatalities                         0
Emergency_Response_Time_Minutes    0
Weather_Condition                  0
SDG_Indicator                      0
Victim_Gender                      0
Victim_Age_Group                   0
Road_User_Type                     0
Total_People_Killed                0
dtype: int64
Total missing cells: 0
Dataset completeness: 100.0%


## Check for Duplicate Rows

In [74]:
full_dups = df.duplicated().sum()
id_dups   = df["Accident_ID"].duplicated().sum()
print(f"Fully duplicate rows:   {full_dups}")
print(f"Duplicate Accident_IDs: {id_dups}")
print(f"Unique IDs: {df['Accident_ID'].nunique()}")

Fully duplicate rows:   0
Duplicate Accident_IDs: 0
Unique IDs: 550


##  Strip Whitespace from Text Columns
Removing invisible leading/trailing spaces that cause silent mismatches.

In [75]:
str_cols = [c for c in df.columns if df[c].dtype == "object" or str(df[c].dtype) == "string"]
for col in str_cols:
    df[col] = df[col].str.strip()
print(f"✓ Stripped {len(str_cols)} string columns.")

✓ Stripped 0 string columns.


##  Validate Coordinates (Lat / Lon)
Nepal lies between **Latitude 26°–30°N** and **Longitude 80°–88°E**. Any coordinates outside this range are invalid.

In [76]:
print("Latitude  range:", df["Latitude"].min(), "to", df["Latitude"].max())
print("Longitude range:", df["Longitude"].min(), "to", df["Longitude"].max())

invalid_lat = df[(df["Latitude"]  < 26) | (df["Latitude"]  > 30)]
invalid_lon = df[(df["Longitude"] < 80) | (df["Longitude"] > 88)]
print(f"Out-of-range Latitude rows:  {len(invalid_lat)}")
print(f"Out-of-range Longitude rows: {len(invalid_lon)}")

Latitude  range: 26.425406 to 28.778103
Longitude range: 80.535828 to 87.690388
Out-of-range Latitude rows:  0
Out-of-range Longitude rows: 0


##  Validate Numeric Columns
Counts like Injuries, Fatalities, and Vehicles cannot be negative.

In [78]:
numeric_cols = [
    "Injuries", "Fatalities", "Number_of_Vehicles_Involved",
    "Emergency_Response_Time_Minutes", "Total_People_Killed"
]
for col in numeric_cols:
    neg = (df[col] < 0).sum()
    print(f"  {col:<35} negatives: {neg}  |  max: {df[col].max()}")

  Injuries                            negatives: 0  |  max: 8
  Fatalities                          negatives: 0  |  max: 4
  Number_of_Vehicles_Involved         negatives: 0  |  max: 4
  Emergency_Response_Time_Minutes     negatives: 0  |  max: 73
  Total_People_Killed                 negatives: 0  |  max: 4


## Mapping age to human understandable term

In [79]:

age_map = {
    '0–14':  'Child (0-14)',
    '15–24': 'Young Adult (15-24)',
    '25–44': 'Adult (25-44)',
    '45–64': 'Middle Aged (45-64)',
    '65+':   'Senior (65+)'
}

df['Victim_Age_Group'] = df['Victim_Age_Group'].map(age_map)

print('Updated Victim_Age_Group:')
print(df['Victim_Age_Group'].value_counts())
print('\nAny nulls:', df['Victim_Age_Group'].isnull().sum())

Updated Victim_Age_Group:
Victim_Age_Group
Adult (25-44)          183
Young Adult (15-24)    181
Middle Aged (45-64)     99
Senior (65+)            47
Child (0-14)            40
Name: count, dtype: int64

Any nulls: 0


## Validate Categorical Columns
Each category column should only contain expected values. Typos create fake categories.

In [80]:
expected = {
    "Road_Type":        ["Highway", "Urban Road", "Hill Road", "Rural Road"],
    "Vehicle_Type":     ["Motorcycle", "Car", "Bus", "Truck", "Tempo", "Bicycle"],
    "Accident_Cause":   ["Human Error", "Overspeeding", "Drunk Driving",
                         "Weather", "Flood", "Landslide", "Mechanical Failure"],
    "Time_of_Day":      ["Morning", "Afternoon", "Evening", "Night"],
    "Weather_Condition":["Clear", "Rainy", "Foggy"],
    "Victim_Gender":    ["Male", "Female", "Other"],
    "Road_User_Type":   ["Motorcyclist", "Driver", "Passenger", "Cyclist"],
}
for col, vals in expected.items():
    unexpected = set(df[col].unique()) - set(vals)
    status = " OK" if not unexpected else f" Unexpected: {unexpected}"
    print(f"  {col:<22} {status}")

  Road_Type               OK
  Vehicle_Type            OK
  Accident_Cause          Unexpected: {'Road Condition'}
  Time_of_Day             OK
  Weather_Condition       OK
  Victim_Gender           OK
  Road_User_Type          OK


##  Create Accident_Date & Month_Name Columns
- **Accident_Date**: proper  column (day=1 since only Year+Month available)
- **Month_Name**: human-readable e.g.  instead of 

In [81]:
# Build datetime from Year + Month (day=1 since day not recorded)
df["Accident_Date"] = pd.to_datetime(df[["Year", "Month"]].assign(Day=1))
df["Month_Name"]    = df["Accident_Date"].dt.month_name()

print("New columns added:")
print(df[["Year", "Month", "Month_Name", "Accident_Date"]].head(4))

New columns added:
   Year  Month Month_Name Accident_Date
0  2022     12   December    2022-12-01
1  2025      7       July    2025-07-01
2  2023     10    October    2023-10-01
3  2023      2   February    2023-02-01


## Sort by Year , Month , Accident_ID
Sorts chronologically. Accident_ID is used as tiebreaker for records in the same month.

In [82]:
df = df.sort_values(
    ["Year", "Month", "Accident_ID"],
    ascending=[True, True, True]
).reset_index(drop=True)

print("First 4 rows after sort:")
print(df[["Accident_ID", "Year", "Month", "Month_Name"]].head(4))
print("Last 3 rows:")
print(df[["Accident_ID", "Year", "Month", "Month_Name"]].tail(3))

First 4 rows after sort:
  Accident_ID  Year  Month Month_Name
0   ACC-00047  2021      1    January
1   ACC-00123  2021      1    January
2   ACC-00152  2021      1    January
3   ACC-00183  2021      1    January
Last 3 rows:
    Accident_ID  Year  Month Month_Name
547   ACC-00463  2025     12   December
548   ACC-00470  2025     12   December
549   ACC-00484  2025     12   December


##   Reorder Columns Logically
Order: **ID , Time , Location , Spatial (Lat/Lon) , Incident Details , Victim Info**

In [83]:
cols = [
    # Identity & Time
    "Accident_ID", "Year", "Month", "Month_Name", "Accident_Date",
    # Location
    "Province", "District", "City / Place",
    # Spatial (critical for clustering)
    "Latitude", "Longitude",
    # Incident details
    "Road_Type", "Vehicle_Type", "Accident_Cause", "Time_of_Day",
    "Weather_Condition", "Number_of_Vehicles_Involved",
    # Impact
    "Injuries", "Fatalities", "Total_People_Killed",
    "Emergency_Response_Time_Minutes",
    # Victim info
    "Victim_Gender", "Victim_Age_Group", "Road_User_Type", "SDG_Indicator"
]
df = df[cols]
print("Final columns:", df.columns.tolist())
print("Shape:", df.shape)

Final columns: ['Accident_ID', 'Year', 'Month', 'Month_Name', 'Accident_Date', 'Province', 'District', 'City / Place', 'Latitude', 'Longitude', 'Road_Type', 'Vehicle_Type', 'Accident_Cause', 'Time_of_Day', 'Weather_Condition', 'Number_of_Vehicles_Involved', 'Injuries', 'Fatalities', 'Total_People_Killed', 'Emergency_Response_Time_Minutes', 'Victim_Gender', 'Victim_Age_Group', 'Road_User_Type', 'SDG_Indicator']
Shape: (550, 24)


## Drop SDC

In [84]:
df = df.drop(columns=['SDG_Indicator'])

## Final Check & Save

In [85]:
# Final check
print("=== FINAL DATASET SUMMARY ===")
print(f"Shape:         {df.shape[0]} rows * {df.shape[1]} columns")
print(f"Missing values:{df.isnull().sum().sum()}")
print(f"Duplicates:    {df.duplicated().sum()}")
print(f"Year range:    {df['Year'].min()} to {df['Year'].max()}")
print(f"Lat range:     {df['Latitude'].min():.4f} to {df['Latitude'].max():.4f}")
print(f"Lon range:     {df['Longitude'].min():.4f} to {df['Longitude'].max():.4f}")
print()
print(df.head(3))

# Save
df.to_csv("../data/Nepal_RoadAccidents_Cleaned.csv", index=False)
print(" Saved: Nepal_RoadAccidents_Cleaned.csv")

=== FINAL DATASET SUMMARY ===
Shape:         550 rows * 23 columns
Missing values:0
Duplicates:    0
Year range:    2021 to 2025
Lat range:     26.4254 to 28.7781
Lon range:     80.5358 to 87.6904

  Accident_ID  Year  Month Month_Name Accident_Date       Province  District  \
0   ACC-00047  2021      1    January    2021-01-01        Madhesh  Dhanusha   
1   ACC-00123  2021      1    January    2021-01-01  Sudurpashchim   Kailali   
2   ACC-00152  2021      1    January    2021-01-01        Madhesh  Dhanusha   

  City / Place   Latitude  Longitude  ... Time_of_Day Weather_Condition  \
0     Janakpur  26.741281  85.901595  ...       Night             Rainy   
1      Tikapur  28.536148  81.114083  ...       Night             Rainy   
2  Lakshminiya  26.682029  85.990802  ...     Morning             Rainy   

  Number_of_Vehicles_Involved Injuries Fatalities  Total_People_Killed  \
0                           1        6          0                    0   
1                           4   